In [1]:
import pandas as pd
import numpy as np
import optuna
from sklearn.metrics import roc_auc_score
import catboost as cb

In [2]:
df_cat = pd.read_csv('df_cat.csv')
df_cat.head()

,age,workclass,education,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,income
0,90,Private,HS-grad,Widowed,Prof-specialty,Not-in-family,White,Female,0.0,8.379539,40,United-States,<=50K
1,82,Private,HS-grad,Widowed,Exec-managerial,Not-in-family,White,Female,0.0,8.379539,18,United-States,<=50K
2,66,Private,Some-college,Widowed,Prof-specialty,Unmarried,Black,Female,0.0,8.379539,40,United-States,<=50K
3,54,Private,7th-8th,Divorced,Machine-op-inspct,Unmarried,White,Female,0.0,8.268988,40,United-States,<=50K
4,41,Private,Some-college,Separated,Prof-specialty,Own-child,White,Female,0.0,8.268988,40,United-States,<=50K


In [3]:
X = df_cat.drop('income', axis=1)
y = df_cat['income']
X.shape, y.shape

((32561, 12), (32561,))

In [4]:
from sklearn.model_selection import train_test_split
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3, random_state=19)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((22792, 12), (9769, 12), (22792,), (9769,))

In [5]:
df_cat_val = X_test.copy()
df_cat_val['target'] = y_test
df_cat_val.head()

,age,workclass,education,marital.status,occupation,relationship,race,sex,capital.gain,capital.loss,hours.per.week,native.country,target
31966,27,Private,Some-college,Never-married,Other-service,Not-in-family,White,Female,0.0,0.0,34,United-States,<=50K
29138,34,Private,Some-college,Separated,Adm-clerical,Unmarried,Black,Female,0.0,0.0,40,Other,<=50K
26720,62,Local-gov,9th,Divorced,Protective-serv,Not-in-family,Black,Male,0.0,0.0,24,United-States,<=50K
8866,26,Private,HS-grad,Never-married,Craft-repair,Own-child,White,Male,0.0,0.0,35,United-States,<=50K
13002,39,Private,10th,Married-civ-spouse,Machine-op-inspct,Husband,White,Male,0.0,0.0,60,United-States,<=50K


In [6]:
df_cat_val.to_csv('df_cat_val.csv', index=False)

In [8]:
target = y.value_counts()
res = target.iloc[0] / target.iloc[1]
print(f'Target disbalance: {res:.3f}')

Target disbalance: 3.153


In [9]:
cat_features = ['workclass', 'education', 'marital.status', 'occupation', 'relationship', 'race',
               'sex', 'native.country']

In [13]:
def objective(trial):

    params = {

        'scale_pos_weight': res,
        
        'iterations': 1000,
        'learning_rate': trial.suggest_float('learning_rate', 1e-3, 0.1, log=True),
        'depth': trial.suggest_int('depth', 4, 10),
        'l2_leaf_reg': trial.suggest_float('l2_leaf_reg', 1e-8, 10.0, log=True),
        'bootstrap_type': 'MVS',
        'subsample': trial.suggest_float('subsample', 0.5, 1.0),
        'random_strength': trial.suggest_float('random_strength', 1e-8, 10.0, log=True),
        'od_type': 'Iter',
        'od_wait': 50,
        'verbose': False,
        'eval_metric': 'AUC',
        'random_state': 19,
    }

    X_t, X_v, y_t, y_v = train_test_split(X_train, y_train, test_size=0.2, random_state=19)

    model = cb.CatBoostClassifier(**params)
    
    model.fit(
        X_t, y_t,
        cat_features=cat_features,
        eval_set=(X_v, y_v),
        early_stopping_rounds=50
    )

    preds = model.predict_proba(X_v)[:, 1]
    roc_auc = roc_auc_score(y_v, preds)
    return roc_auc

study = optuna.create_study(direction='maximize')
study.optimize(objective, n_trials=40)

[I 2026-03-19 17:44:48,918] A new study created in memory with name: no-name-85a2098d-560f-43bb-b6b9-04e1f284f7cb
[I 2026-03-19 17:45:28,904] Trial 0 finished with value: 0.922686630712461 and parameters: {'learning_rate': 0.006379862586480066, 'depth': 4, 'l2_leaf_reg': 0.06538874506293232, 'subsample': 0.6639570265724569, 'random_strength': 4.097351996948309e-08}. Best is trial 0 with value: 0.922686630712461.
[I 2026-03-19 17:45:45,896] Trial 1 finished with value: 0.9267375065345546 and parameters: {'learning_rate': 0.06710454343318639, 'depth': 6, 'l2_leaf_reg': 3.936631906461959e-07, 'subsample': 0.7115159780013379, 'random_strength': 4.016361572368425}. Best is trial 1 with value: 0.9267375065345546.
[I 2026-03-19 17:47:13,519] Trial 2 finished with value: 0.9190193346097404 and parameters: {'learning_rate': 0.0024370349653439318, 'depth': 9, 'l2_leaf_reg': 8.494397716259544e-07, 'subsample': 0.8530158564138532, 'random_strength': 3.261540471280482e-08}. Best is trial 1 with val

In [14]:
best_params = study.best_params
best_params

{'learning_rate': 0.022172895478994845,
 'depth': 4,
 'l2_leaf_reg': 0.0019445029163544026,
 'subsample': 0.5019863453494742,
 'random_strength': 0.07143726229129845}

In [15]:
print(f'ROC AUC score :{study.best_value:.4f}')

ROC AUC score :0.9295


In [16]:
cat_model = cb.CatBoostClassifier(**best_params, random_state=19)

In [19]:
cat_model.fit(X_train, y_train, cat_features=cat_features, verbose=0)

In [20]:
import joblib
joblib.dump(cat_model, 'cat_model.joblib')

['cat_model.joblib']